In [ ]:
## beginning

In [ ]:
import sys
import os.path
import logging
import argparse

import numpy as np
from datetime import datetime
from collections import OrderedDict

import torch
import torch.nn as nn

from utils import utils_logger
from utils import utils_model
from utils import utils_image as util

from models import basicblock as B

In [ ]:
sys.argv = ['']
parser = argparse.ArgumentParser()
parser.add_argument('--model_name', type=str, default='dncnn_50', help='dncnn_15, dncnn_25, dncnn_50, dncnn_gray_blind, dncnn_color_blind, dncnn3')
parser.add_argument('--testset_name', type=str, default='set12', help='set5, set12, real_faces')
parser.add_argument('--noise_level_img', type=int, default=15, help='noise level: 15, 25, 50')
parser.add_argument('--x8', type=bool, default=False, help='x8 to boost performance')
parser.add_argument('--show_img', type=bool, default=False, help='show the image')
parser.add_argument('--model_pool', type=str, default='model_zoo', help='path of model_zoo')
parser.add_argument('--testsets', type=str, default='testsets', help='path of testing folder')
parser.add_argument('--results', type=str, default='results', help='path of results')
parser.add_argument('--need_degradation', type=bool, default=True, help='add noise or not')
parser.add_argument('--task_current', type=str, default='dn', help='dn for denoising, fixed!')
parser.add_argument('--sf', type=int, default=1, help='unused for denoising')
args = parser.parse_args()

In [ ]:
if ('color' in args.model_name):
    n_channels = 3 # fixed, 1 for grayscale image, 3 for color image
else:
    n_channels = 1 # fixed for grayscale image
if args.model_name in ['dncnn_gray_blind', 'dncnn_color_blind', 'dncnn3']:
    nb = 20               # fixed, total number of conv layers
else:
    nb = 17               # fixed, total number of conv layers

In [ ]:
result_name = args.testset_name + '_' + args.model_name     # fixed
border = args.sf if args.task_current == 'sr' else 0        # shave boader to calculate PSNR and SSIM
model_path = os.path.join(args.model_pool, args.model_name+'.pth')
print(result_name)
print(model_path)

In [ ]:
# ----------------------------------------
# L_path, E_path, H_path
# ----------------------------------------
L_path = os.path.join(args.testsets, args.testset_name)  # L_path, for Low-quality images
H_path = L_path                                          # H_path, for High-quality images
E_path = os.path.join(args.results, result_name)         # E_path, for Estimated images

print(L_path)
print(H_path)
print(E_path)

In [ ]:
util.mkdir(E_path)
if (H_path == L_path): args.need_degradation = True
logger_name = result_name
log_path=os.path.join(E_path, logger_name+'.log')
utils_logger.logger_info(logger_name, log_path=log_path)
logger = logging.getLogger(logger_name)
print(log_path)

In [ ]:
need_H = True if H_path is not None else False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
class DnCNN(nn.Module):
    def __init__(self, in_nc=1, out_nc=1, nc=64, nb=17, act_mode='BR'):
        """
        # ------------------------------------
        in_nc: channel number of input
        out_nc: channel number of output
        nc: channel number
        nb: total number of conv layers
        act_mode: batch norm + activation function; 'BR' means BN+ReLU.
        # ------------------------------------
        Batch normalization and residual learning are
        beneficial to Gaussian denoising (especially
        for a single noise level).
        The residual of a noisy image corrupted by additive white
        Gaussian noise (AWGN) follows a constant
        Gaussian distribution which stablizes batch
        normalization during training.
        # ------------------------------------
        """
        super(DnCNN, self).__init__()
        assert 'R' in act_mode or 'L' in act_mode, 'Examples of activation function: R, L, BR, BL, IR, IL'
        bias = True

        m_head = B.conv(in_nc, nc, mode='C'+act_mode[-1], bias=bias)
        m_body = [B.conv(nc, nc, mode='C'+act_mode, bias=bias) for _ in range(nb-2)]
        m_tail = B.conv(nc, out_nc, mode='C', bias=bias)

        self.model = B.sequential(m_head, *m_body, m_tail)

    def forward(self, x):
        n = self.model(x)
        return x-n

In [ ]:
# ----------------------------------------
# load model
# ----------------------------------------
model = DnCNN(in_nc=n_channels, out_nc=n_channels, nc=64, nb=nb, act_mode='R')
# model = net(in_nc=n_channels, out_nc=n_channels, nc=64, nb=nb, act_mode='BR')  # use this if BN is not merged by utils_bnorm.merge_bn(model)
model.load_state_dict(torch.load(model_path), strict=True)
model.eval()
for k, v in model.named_parameters(): v.requires_grad = False
model = model.to(device)
print(model)
logger.info('Model path: {:s}'.format(model_path))
number_parameters = sum(map(lambda x: x.numel(), model.parameters()))
logger.info('Params number: {}'.format(number_parameters))

In [ ]:
test_results = OrderedDict()
test_results['psnr'] = []
test_results['ssim'] = []

In [ ]:
logger.info('model_name:{}, image sigma:{}'.format(args.model_name, args.noise_level_img))
logger.info(L_path)
L_paths = util.get_image_paths(L_path)
H_paths = util.get_image_paths(H_path) if need_H else None

In [ ]:
for (idx, img) in enumerate(L_paths):
    # ------------------------------------
    # (1) img_L
    # ------------------------------------
    img_name, ext = os.path.splitext(os.path.basename(img))
    # logger.info('{:->4d}--> {:>10s}'.format(idx+1, img_name+ext))
    img_L = util.imread_uint(img, n_channels=n_channels)
    img_L = util.uint2single(img_L)

    if args.need_degradation: # degradation process
        np.random.seed(seed=0) # for reproducibility
        img_L += np.random.normal(0, args.noise_level_img/255., img_L.shape)

    util.imshow(util.single2uint(img_L), title='Noisy image with noise level {}'.format(args.noise_level_img)) if args.show_img else None

    img_L = util.single2tensor4(img_L)
    img_L = img_L.to(device)

    # ------------------------------------
    # (2) img_E
    # ------------------------------------
    if not args.x8:
        img_E = model(img_L)
    else:
        img_E = utils_model.test_mode(model, img_L, mode=3)

    img_E = util.tensor2uint(img_E)
    img_L = util.tensor2uint(img_L)

    util.imsave(img_E, os.path.join(E_path, img_name+'_E'+ext))
    util.imsave(img_L, os.path.join(E_path, img_name+'_L'+ext))

    if need_H:
        # --------------------------------
        # (3) img_H
        # --------------------------------

        img_H = util.imread_uint(H_paths[idx], n_channels=n_channels)
        img_H = img_H.squeeze()

        # --------------------------------
        # PSNR and SSIM
        # --------------------------------

        psnr = util.calculate_psnr(img_E, img_H, border=border)
        ssim = util.calculate_ssim(img_E, img_H, border=border)
        test_results['psnr'].append(psnr)
        test_results['ssim'].append(ssim)
        logger.info('{:s} - PSNR: {:.2f} dB; SSIM: {:.4f}.'.format(img_name+ext, psnr, ssim))
        util.imshow(np.concatenate([img_E, img_H], axis=1), title='Recovered / Ground-truth') if args.show_img else None
        util.imsave(img_H, os.path.join(E_path, img_name+'_H'+ext))

In [ ]:
if need_H:
    ave_psnr = sum(test_results['psnr']) / len(test_results['psnr'])
    ave_ssim = sum(test_results['ssim']) / len(test_results['ssim'])
    logger.info('Average PSNR/SSIM(RGB) - {} - PSNR: {:.2f} dB; SSIM: {:.4f}'.format(result_name, ave_psnr, ave_ssim))

In [ ]:
# end